# Public Health Surveillance Data Pipeline
**Author:** Aly Drame, MD, MPH, MBA

## What This Pipeline Does
Transforms raw, messy surveillance case data into clean, validated,
analysis-ready outputs — mirroring workflows performed in CDC DCIPHER.

**Pipeline stages:**
1. EXTRACT — Load all four raw tables
2. VALIDATE — Run all data quality checks (before cleaning)
3. CLEAN — Standardize dates, remove duplicates, fix coded values
4. TRANSFORM — Merge tables, add derived variables
5. AGGREGATE — Calculate weekly counts, jurisdiction summaries
6. EXPORT — Generate visualizations and automated Word report

**All data are synthetic — generated for portfolio demonstration.**

In [ ]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Run data generation if raw files don't exist
if not os.path.exists('../data/raw/cases_raw.csv'):
    print('Generating synthetic data...')
    exec(open('../data/synthetic/generate_synthetic_data.py').read())

## Step 1: Extract

In [ ]:
from src.extract import load_all_raw_tables
tables = load_all_raw_tables('../data/raw')
cases         = tables['cases']
lab_results   = tables['lab_results']
contacts      = tables['contacts']
jurisdictions = tables['jurisdictions']
print(f"cases: {cases.shape} | lab: {lab_results.shape} | "
      f"contacts: {contacts.shape} | jurisdictions: {jurisdictions.shape}")

## Step 2: Validate (Before Cleaning)

In [ ]:
from src.validate import run_all_validations
validation_results = run_all_validations(tables, '../outputs')
miss_before = validation_results['missingness_cases']

## Step 3: Clean

In [ ]:
from src.clean import clean_cases, clean_lab_results, clean_contacts
cases_clean    = clean_cases(cases)
lab_clean      = clean_lab_results(lab_results)
contacts_clean = clean_contacts(contacts)

## Step 4: Validate After Cleaning

In [ ]:
from src.validate import missingness_report
miss_after = missingness_report(cases_clean, 'cases_clean',
                                '../outputs/missingness_after.csv')

## Step 5: Transform — Build Analytical Dataset

In [ ]:
from src.transform import build_analytical_dataset, save_processed_tables
analytical = build_analytical_dataset(
    cases_clean, lab_clean, contacts_clean, jurisdictions)
save_processed_tables(cases_clean, lab_clean, contacts_clean,
                       analytical, '../data/processed')
analytical.head(3)

## Step 6: Aggregate

In [ ]:
from src.aggregate import (weekly_case_counts, jurisdiction_summary,
                             contact_tracing_metrics)
weekly = weekly_case_counts(
    analytical, '../outputs/weekly_case_summary.csv')
jur_summary = jurisdiction_summary(
    analytical, jurisdictions, '../outputs/jurisdiction_summary.csv')
contact_metrics = contact_tracing_metrics(
    contacts_clean, cases_clean,
    '../outputs/contact_tracing_metrics.csv')
print(f"Weekly summary: {len(weekly):,} rows")
print(f"Jurisdiction summary: {len(jur_summary):,} rows")
print(f"Contact metrics: {len(contact_metrics):,} rows")

## Step 7: Export Visualizations and Word Report

In [ ]:
from src.export_report import (plot_epidemic_curve,
                                 plot_completeness_heatmap,
                                 plot_disease_distribution,
                                 create_word_report)

plot_epidemic_curve(weekly, output_path='../outputs/epidemic_curve.png')
plot_completeness_heatmap(miss_after,
                           output_path='../outputs/completeness_heatmap.png')
plot_disease_distribution(cases_clean,
                           output_path='../outputs/disease_distribution.png')
create_word_report(
    weekly, miss_after, jur_summary,
    epidemic_curve_path='../outputs/epidemic_curve.png',
    output_path='../outputs/weekly_surveillance_report.docx'
)

print('Pipeline complete. All outputs saved to outputs/')

## Pipeline Summary

In [ ]:
print('=== PIPELINE SUMMARY ===')
print(f"Raw records loaded:          {len(cases):,}")
print(f"Records after deduplication: {len(cases_clean):,}")
print(f"Lab results:                 {len(lab_clean):,}")
print(f"Contacts:                    {len(contacts_clean):,}")
print(f"Analytical dataset:          {len(analytical):,} x {analytical.shape[1]}")
print(f"Weeks covered:               {weekly['epi_week_label'].nunique()}")
print(f"Diseases tracked:            {analytical['disease_name'].nunique()}")
print(f"Jurisdictions covered:       {analytical['jurisdiction_id'].nunique()}")